# 02 E3: tuning, overfitting and split sensitivity (train -> dev)

Fixed features: LFCC, 20 coefficients + delta + delta-delta, mean+std (120 columns). Seed 0.
Models: logistic regression, random forest, HistGradientBoosting (all class-balanced).

1. **Random search** (30 settings per model). Each setting is scored on dev and by
   speaker-grouped 5-fold CV on train. Full table: `results/e3_search.csv`.
2. **Selection** by dev and, separately, by grouped CV (split sensitivity), plus untuned defaults.
3. **Gap table**: train vs grouped-CV vs dev EER (eval is added later, once per final model).
4. **Learning curves**: dev-selected settings at 5/10/25/50/100% of train.

Final rows with 95% CIs: `results/e3.csv`. Figure: `results/figures/e3_tuning.png`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import loguniform

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.features import load_xy
from deepvoice.models import random_search, run

FS = {"groups": ("lfcc",), "n_coef": 20, "deltas": 2, "stats": ("mean", "std")}
train, dev = load_xy("train", **FS), load_xy("dev", **FS)
X = list(train.columns[4:])
print(train.shape, dev.shape, len(X), "features")

## 1. Random search

In [ ]:
SPACES = {
    "logreg": ({"C": loguniform(1e-4, 1e2), "penalty": ["l1", "l2"]}, {"solver": "liblinear"}),
    "rf": ({"n_estimators": [100, 200, 400], "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 2, 5, 10], "max_features": ["sqrt", "log2", 0.3]}, {"n_jobs": 6}),
    "hgb": ({"learning_rate": loguniform(0.02, 0.3), "max_depth": [None, 3, 5, 8],
             "l2_regularization": loguniform(1e-3, 10), "max_iter": [100, 200, 400]},
            {"early_stopping": False}),
}
DEFAULTS = {"logreg": {"C": 1.0}, "rf": {"n_estimators": 300, "n_jobs": 6}, "hgb": {}}

search_csv = ROOT / "results/e3_search.csv"
if search_csv.exists():  # computed once; delete the file on purpose to redo the search
    search = pd.read_csv(search_csv)
else:
    search = pd.concat([random_search(m, space, train, dev, X, n_iter=30, seed=0, fixed=fixed)
                        for m, (space, fixed) in SPACES.items()], ignore_index=True)
    search.to_csv(search_csv, index=False, float_format="%.6g")
search.groupby("model")[["dev_eer", "cv_eer", "fit_time"]].describe().round(4).T

## 2. Selection on dev vs on grouped CV, plus untuned defaults

In [ ]:
picks = {}
for m in SPACES:
    s = search[search.model == m]
    picks[(m, "default")] = DEFAULTS[m]
    picks[(m, "dev")] = json.loads(s.loc[s.dev_eer.idxmin(), "params"])
    picks[(m, "cv")] = json.loads(s.loc[s.cv_eer.idxmin(), "params"])
    print(f"{m:7s} same setting chosen by dev and CV: {picks[(m, 'dev')] == picks[(m, 'cv')]}")

for (m, sel), params in picks.items():
    run(1, m, X, train, [("asvspoof", "train", train), ("asvspoof", "dev", dev)],
        params=params, experiment="e3", feature_set=f"lfcc20dd_{sel}sel" if sel != "default" else "lfcc20dd_default")

## 3. Gap table: train vs speaker-grouped CV vs dev (EER %)

In [ ]:
from deepvoice.models import grouped_cv_eer

r = pd.read_csv(ROOT / "results/e3.csv")
rows = []
for (m, sel), params in picks.items():
    fs = f"lfcc20dd_{sel}sel" if sel != "default" else "lfcc20dd_default"
    rr = r[r.config == f"e3_{m}_{fs}_s0"].set_index("split")
    if sel == "default":
        cv = grouped_cv_eer(m, params, train, X)
    else:
        s = search[(search.model == m) & (search.params == json.dumps(params))]
        cv = s.cv_eer.iloc[0]
    rows.append({"model": m, "selected by": sel, "train": 100 * rr.loc["train", "eer"], "grouped CV": 100 * cv,
                 "dev": 100 * rr.loc["dev", "eer"],
                 "dev 95% CI": f"{100*rr.loc['dev','ci_low']:.2f}-{100*rr.loc['dev','ci_high']:.2f}",
                 "fit s": rr.loc["dev", "train_time"], "params": json.dumps(params)})
gap = pd.DataFrame(rows)
gap.to_csv(ROOT / "results/e3_gap.csv", index=False, float_format="%.4g")
gap.round(3)

## 4. Learning curves (dev-selected settings)

In [ ]:
FRACS = [0.05, 0.10, 0.25, 0.50, 1.00]
for m in SPACES:
    for f in FRACS:
        sub = train if f == 1 else train.groupby("label", group_keys=False).sample(frac=f, random_state=0)
        run(1, m, X, sub, [("asvspoof", "dev", dev)], params=picks[(m, "dev")],
            experiment="e3", feature_set=f"lfcc20dd_devsel_frac{int(f * 100):03d}")
lc = pd.read_csv(ROOT / "results/e3.csv")
lc = lc[lc.config.str.contains("_frac")].copy()
lc["frac"] = lc.config.str.extract(r"frac(\d+)").astype(int)[0]
lc["model"] = lc.model.str.split(":").str[0]
lc.pivot_table(index="frac", columns="model", values="eer").mul(100).round(2)

## Figure

In [ ]:
COL = {"logreg": "#1baf7a", "rf": "#6250d6", "hgb": "#eda100"}
NAME = {"logreg": "logistic regression", "rf": "random forest", "hgb": "gradient boosting"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# (a) learning curves with CI band
for m in SPACES:
    d = lc[lc.model == m].sort_values("frac")
    ax[0].plot(d.frac, 100 * d.eer, marker="o", ms=6, lw=2, color=COL[m], label=NAME[m])
    ax[0].fill_between(d.frac, 100 * d.ci_low, 100 * d.ci_high, color=COL[m], alpha=0.15, lw=0)
ax[0].set_xscale("log"); ax[0].set_xticks([5, 10, 25, 50, 100], ["5%", "10%", "25%", "50%", "100%"])
ax[0].set_xlabel("share of train used"); ax[0].set_ylabel("dev EER % (band = 95% CI)")
ax[0].grid(axis="y", color=GRID, linewidth=0.6); ax[0].legend(frameon=False)
ax[0].set_title("(a) Learning curves (dev-selected settings)", loc="left", color=INK)

# (b) gap: train / grouped CV / dev for the dev-selected models
g = gap[gap["selected by"] == "dev"].set_index("model")
marks = [("train", "o", "train"), ("grouped CV", "s", "speaker-grouped CV"), ("dev", "D", "dev")]
for i, m in enumerate(SPACES):
    vals = [g.loc[m, c] for c, _, _ in marks]
    ax[1].plot([min(vals), max(vals)], [i, i], color=COL[m], lw=2)
    for (c, mk, _), v in zip(marks, vals):
        ax[1].plot(v, i, marker=mk, ms=8, color=COL[m], mec="white", mew=1.5, ls="none")
for c, mk, lab in marks:
    ax[1].plot([], [], marker=mk, ms=7, color=MUTED, ls="none", label=lab)
ax[1].set_yticks(range(len(SPACES)), [NAME[m] for m in SPACES]); ax[1].invert_yaxis()
ax[1].set_xlabel("EER %"); ax[1].grid(axis="x", color=GRID, linewidth=0.6)
ax[1].legend(frameon=False, loc="lower right")
ax[1].set_title("(b) Train vs grouped CV vs dev (tuned on dev)", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e3_tuning.png", dpi=200)
fig.savefig(ROOT / "results/figures/e3_tuning.pdf")